In [ ]:
Cella da eseguire solo la prima volta prima di creare la SparkSession: poi riavvare il kernel !!!

In [ ]:
import os
import urllib.request

jars = {
    "hadoop-aws-3.3.4.jar": "https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar",
    "aws-java-sdk-bundle-1.12.262.jar": "https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar"
}

for filename, url in jars.items():
    dest = f"/tmp/spark_jars/{filename}"
    if not os.path.exists(dest):
        print(f"⬇️  Downloading {filename}...")
        urllib.request.urlretrieve(url, dest)
        print(f"✅ {filename}")
    else:
        print(f"⏭️  Already exists: {filename}")

In [ ]:
import os
# Imposta parametri di avvio di PySpark PRIMA della creazione della SparkSession
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    # --jars: aggiunge manualmente librerie Java (JAR) a Spark
    # hadoop-aws: abilita il supporto al protocollo s3a:// (S3 / MinIO)
    "--jars /tmp/spark_jars/hadoop-aws-3.3.4.jar,"
    # aws-java-sdk: client AWS usato da Spark per comunicare con S3/MinIO
    "/tmp/spark_jars/aws-java-sdk-bundle-1.12.262.jar "
    # pyspark-shell: indica che questi argomenti si applicano alla sessione PySpark
    "pyspark-shell"
)

from functools import reduce
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, substring, regexp_replace, trim, avg,
    when, round as spark_round, count,
    min as spark_min, max as spark_max, sum as spark_sum
)

BRONZE_PATH = "s3a://rental-observatory/bronze/"
SILVER_PATH = "s3a://rental-observatory/silver/"
GOLD_PATH   = "s3a://rental-observatory/gold/"

spark = SparkSession.builder \
    .appName("NYC_Rental_Stress") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f" Spark {spark.version}\n")


In [ ]:


# NYC ZIP code prefixes (5 boroughs only)
# Excludes Long Island, Hamptons, Upstate NY
NYC_ZIP_PREFIXES = ["100", "101", "102", "103", "104",  # Manhattan + Bronx
                    "111", "112", "113", "114", "116",  # Brooklyn + Queens
                    "103"]                               # Staten Island

# ============================================================
# 2. LOAD RAW DATA FROM BRONZE LAYER
# ============================================================
print("=" * 55)
print(" STEP 1 — LOADING DATA (Bronze → Spark)")
print("=" * 55)

pop_df = spark.read.csv(
    BRONZE_PATH + "ACSDT5Y2024.B01003-Data.csv",
    header=True, inferSchema=True)

income_df = spark.read.csv(
    BRONZE_PATH + "ACSDT5Y2024.B19013-Data.csv",
    header=True, inferSchema=True)

zillow_df = spark.read.csv(
    BRONZE_PATH + "Zip_zori_uc_sfrcondomfr_sm_month.csv",
    header=True, inferSchema=True)

listings_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .load(BRONZE_PATH + "listings_NY.csv.gz")

calendar_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .load(BRONZE_PATH + "calendar_NY.csv.gz")

print(f"  Population:  {pop_df.count():,} rows")
print(f"  Income:      {income_df.count():,} rows")
print(f"  Zillow:      {zillow_df.count():,} rows")
print(f"  Listings:    {listings_df.count():,} rows")
print(f"  Calendar:    {calendar_df.count():,} rows")
print("✅ Bronze loaded.\n")

In [ ]:
pop_df.printSchema()
pop_df.show(5)
pop_df.dtypes

# Popolazione per ZIP
census_population = (
    pop_df  # parto dal DataFrame originale (bronze)
    .filter(col("GEO_ID") != "Geography")  # rimuovo la riga header duplicata/non valida
    .withColumn(
        "zip_code",
        substring(col("GEO_ID"), -5, 5)  # estraggo gli ultimi 5 caratteri → ZIP code
    )
    .withColumn(
        "total_population",  # nuova colonna con la popolazione totale
        when(
            regexp_replace(col("B01003_001E"), "[^0-9]", "") == "",  # se dopo pulizia non resta nulla
            None  # allora metto valore nullo
        ).otherwise(
            regexp_replace(col("B01003_001E"), "[^0-9]", "")  # rimuovo tutto ciò che non è numero
            .cast("int")  # converto a intero
        )
    )
    .select("zip_code", "total_population")  # tengo solo le colonne utili
    .dropna()  # elimino righe con valori nulli
)
census_population.show()

In [ ]:
income_df.printSchema()

# Reddito mediano per ZIP
census_income = (
    income_df  # parto dal DataFrame originale (bronze)
    .filter(col("GEO_ID") != "Geography")  # rimuovo la riga header duplicata/non valida
    .withColumn("zip_code", substring(col("GEO_ID"), -5, 5))  # estraggo ZIP code
    .withColumn("median_income",  # nuova colonna reddito mediano
        when(regexp_replace(col("B19013_001E"), "[^0-9]", "") == "", None)  # se vuoto → null
        .otherwise(regexp_replace(col("B19013_001E"), "[^0-9]", "").cast("int"))  # pulisco e casto a int
    )
    .select("zip_code", "median_income")  # tengo solo colonne utili
    .dropna()  # rimuovo righe con null
)
census_income.show(5)

In [ ]:
zillow_df.printSchema()
# Zillow ZORI — affitto di mercato NY
zillow_rent = (
    zillow_df
    .filter(col("State") == "NY")  # tengo solo lo stato New York
    .select(
        col("RegionName").cast("string").alias("zip_code"),  # ZIP code (cast a string + rename)
        col("2026-02-28").cast("float").alias("market_rent")  # affitto stimato a fine 2024
    )
    .dropna()  # rimuovo righe con valori null
)
zillow_rent.show(5)

In [ ]:
zillow_rent.select("State").distinct().show(160)


In [53]:
print(f"  Population silver: {census_population.count():,} ZIP")
print(f"  Income silver:     {census_income.count():,} ZIP")
print(f"  Zillow silver:     {zillow_rent.count():,} ZIP")

  Population silver: 33,772 ZIP
  Income silver:     30,547 ZIP
  Zillow silver:     354 ZIP


In [70]:
# ============================================================
# 4. SILVER — AIRBNB LISTINGS
#
# price e estimated_revenue_l365d sono NULL in tutto il dataset NY.
# Usiamo estimated_occupancy_l365d per calcolare l'occupancy_rate.
# Il prezzo sarà approssimato con Zillow ZORI nel Gold layer.
# ============================================================
print("\n--- Airbnb Silver ---")

airbnb_silver = (
    listings_df
    # Rimuove righe dove l'ID non è un numero valido (righe corrotte)
    .filter(col("id").cast("bigint").isNotNull())
    # Rimuove listing senza borough (non possiamo geolocalizzarli)
    .filter(col("neighbourhood_group_cleansed").isNotNull())
    # Rimuove listing senza coordinate (non mappabili)
    .filter(col("latitude").isNotNull())
    .filter(col("longitude").isNotNull())

    # Crea colonna float dei giorni occupati negli ultimi 365 giorni
    # Es: "240" (stringa) → 240.0 (float)
    .withColumn(
        "occupancy_days",
        col("estimated_occupancy_l365d").cast("float")
    )

    # 🛠 FIX 1: Calcolo diretto senza when
    # Spark gestisce già i NULL automaticamente (NULL → NULL)
    # Es: 240 giorni occupati → 240/365 = 0.6575
    .withColumn(
        "occupancy_rate",
        spark_round(col("occupancy_days") / 365.0, 4)
    )

    # 🛠 FIX 2: Convertiamo gli 0.0 sospetti in NULL
    # Motivo: nel dataset, 0 spesso significa "dato mancante"
    # Evita di distorcere medie e analisi nel Gold layer
    .withColumn(
        "occupancy_rate",
        when(col("occupancy_rate") == 0, None)
        .otherwise(col("occupancy_rate"))
    )

    .select(
        col("id").cast("bigint"),                          # ID univoco del listing
        "neighbourhood_group_cleansed",                    # Borough: Manhattan, Brooklyn, Queens, Bronx, Staten Island
        "neighbourhood_cleansed",                          # Quartiere specifico: es. "Williamsburg"
        col("latitude").cast("float"),                     # Latitudine per la heatmap
        col("longitude").cast("float"),                    # Longitudine per la heatmap
        "occupancy_rate",                                  # Tasso occupazione 0.0→1.0 (NULL = dato mancante)
        "occupancy_days",                                  # Giorni occupati raw
        col("room_type"),                                  # Entire home / Private room / etc.
        col("accommodates").cast("int"),                   # Numero massimo ospiti
        col("calculated_host_listings_count").cast("int")  # Listings per host → proxy host commerciale
    )
)

airbnb_count = airbnb_silver.count()
print(f"  Listings Airbnb (silver): {airbnb_count:,}")
print("✅ Airbnb Silver completato.\n")

airbnb_silver.select("neighbourhood_group_cleansed","occupancy_rate").show()


--- Airbnb Silver ---
  Listings Airbnb (silver): 36,445
✅ Airbnb Silver completato.

+----------------------------+--------------+
|neighbourhood_group_cleansed|occupancy_rate|
+----------------------------+--------------+
|                   Manhattan|          NULL|
|                    Brooklyn|        0.6575|
|                   Manhattan|        0.1644|
|                   Manhattan|        0.6575|
|                    Brooklyn|        0.6986|
|                    Brooklyn|          NULL|
|                    Brooklyn|          NULL|
|                   Manhattan|          NULL|
|                    Brooklyn|        0.1644|
|                   Manhattan|          NULL|
|                   Manhattan|          NULL|
|                   Manhattan|          NULL|
|                   Manhattan|          NULL|
|                    Brooklyn|        0.1644|
|                       Bronx|          NULL|
|                    Brooklyn|          NULL|
|                   Manhattan|         

In [ ]:
# ============================================================
# 4b. SILVER — CALENDAR
# ============================================================
print("\n--- Calendar Silver ---")

from pyspark.sql.functions import to_date

calendar_silver = (
    calendar_df
    # Rimuove righe con listing_id non valido
    .filter(col("listing_id").cast("bigint").isNotNull())
    # Cast a bigint per poter fare JOIN con airbnb_silver.id
    .withColumn("listing_id", col("listing_id").cast("bigint"))
    # Converte la stringa data "2025-12-06" in tipo Date di Spark
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
    # 1 se il listing è disponibile quel giorno, 0 altrimenti
    .withColumn(
        "is_available",
        when(col("available") == "t", 1).otherwise(0)
    )
    # NON disponibile (occupato + blocchi host)
    .withColumn(
        "is_blocked",
        when(col("available") == "f", 1).otherwise(0)
    )
    .select("listing_id", "date", "is_available", "is_blocked")
)

# Aggrega per listing
calendar_agg = (
    calendar_silver
    .groupBy("listing_id")
    .agg(
        spark_sum("is_available").alias("days_available"),
        spark_sum("is_blocked").alias("days_blocked"),
        count("date").alias("total_days")
    )
    .filter(col("total_days") >= 300)
    .withColumn(
        "cal_occupancy_rate",
        spark_round(col("days_blocked") / col("total_days"), 4)
    )
)

# Statistiche finali
airbnb_count = airbnb_silver.count()
cal_count = calendar_agg.count()

print(f"  Listings Airbnb (silver):    {airbnb_count:,}")
print(f"  Listing con dati calendar:   {cal_count:,}")

airbnb_silver.select(
    spark_round(avg("occupancy_rate"), 3).alias("avg_occupancy_rate"),
    spark_min("occupancy_rate").alias("min"),
    spark_max("occupancy_rate").alias("max")
).show()

print("✅ Calendar Silver completato.\n")

In [ ]:
calendar_agg.show(5)

In [ ]:
calendar_silver.show(5)

In [ ]:
calendar_df.show(5)

In [ ]:
"""# ============================================================
# 4c. SILVER — AIRBNB LISTINGS ENRICHED
#
# JOIN tra airbnb_silver e calendar_agg per arricchire
# ogni listing con i dati reali di occupazione dal calendar.
# Usiamo left join per mantenere tutti i listing anche se
# non hanno dati nel calendar (cal_occupancy_rate → NULL)
# ============================================================
print("\n--- Airbnb Listings Enriched ---")

airbnb_listings_enriched = (
    airbnb_silver
    # Left join: teniamo tutti i listing di airbnb_silver
    # anche se non hanno dati nel calendar
    .join(calendar_agg, airbnb_silver["id"] == calendar_agg["listing_id"], "left")
    # Usiamo cal_occupancy_rate dal calendar se disponibile,
    # altrimenti fallback su occupancy_rate da estimated_occupancy_l365d
    .withColumn(
        "final_occupancy_rate",
        when(col("cal_occupancy_rate").isNotNull(), col("cal_occupancy_rate"))
        .otherwise(col("occupancy_rate"))
    )
    .select(
        col("id"),
        "neighbourhood_group_cleansed",              # Borough
        "neighbourhood_cleansed",                    # Quartiere
        col("latitude"),
        col("longitude"),
        "room_type",
        "accommodates",
        "calculated_host_listings_count",            # Proxy host commerciale
        "occupancy_rate",                            # Stima Inside Airbnb
        "cal_occupancy_rate",                        # Dato reale dal calendar
        "final_occupancy_rate",                      # Quello che useremo nel Gold
        "days_available",                            # Giorni liberi nell'anno
        "days_blocked",                              # Giorni occupati/bloccati
        "total_days"                                 # Totale giorni nel dataset
    )
)

enriched_count = airbnb_listings_enriched.count()
print(f"  Listings enriched: {enriched_count:,}")
airbnb_listings_enriched.select(
    "neighbourhood_group_cleansed",
    "final_occupancy_rate",
    "cal_occupancy_rate",
    "occupancy_rate"
).show(5)
print("✅ Airbnb Listings Enriched completato.\n")"""

In [ ]:
# Prendi 10 ID da airbnb_silver
airbnb_ids = [row.id for row in airbnb_silver.select("id").limit(10).collect()]
print("Airbnb IDs:", airbnb_ids)

# Cerca quegli stessi ID nel calendar_agg
calendar_agg.filter(col("listing_id").isin(airbnb_ids)).show()

In [ ]:
# Quando è stato scrapeato il calendar?
calendar_df.select("date").distinct().orderBy("date").show(5)

# Quando è stato scrapeato il listings?
listings_df.select("last_scraped").distinct().show(5)

In [ ]:
# Quanti ID ha il calendar?
cal_ids = calendar_df.select("listing_id").distinct().count()
listing_ids = listings_df.select("id").distinct().count()
print(f"Calendar IDs: {cal_ids:,}")
print(f"Listings IDs: {listing_ids:,}")

# Quanti in comune?
cal_df2 = calendar_df.select(col("listing_id").cast("bigint").alias("id"))
common = cal_df2.join(listings_df.select("id"), "id", "inner")
print(f"ID in comune: {common.distinct().count():,}")

In [54]:
# ============================================================
# AIRBNB LISTINGS ENRICHED
#
# Il calendar disponibile non è allineato con i listings
# (0 ID in comune tra i due dataset).
# Si utilizza estimated_occupancy_l365d come proxy
# dell'occupancy rate, fornito direttamente da Inside Airbnb.
# ============================================================

airbnb_listings_enriched = (
    airbnb_silver
    .withColumnRenamed("occupancy_rate", "final_occupancy_rate")
)

print(f"  Listings enriched: {airbnb_listings_enriched.count():,}")
print("✅ Airbnb Listings Enriched completato.\n")

  Listings enriched: 36,445
✅ Airbnb Listings Enriched completato.



In [71]:
# ============================================================
# 5. SAVE SILVER LAYER TO MINIO
#
# Salviamo i dataframe puliti in formato Parquet su MinIO.
# Parquet è un formato colonnare ottimizzato per Spark:
# - compressione migliore dei CSV
# - lettura più veloce (legge solo le colonne necessarie)
# - mantiene i tipi di dato (no problemi di cast al prossimo load)
# ============================================================
print("=" * 55)
print("💾 SAVING SILVER LAYER TO MINIO")
print("=" * 55)

# Census Population Silver
census_population.write \
    .mode("overwrite") \
    .parquet(SILVER_PATH + "census_population/")
print("✅ census_population saved")

# Census Income Silver
census_income.write \
    .mode("overwrite") \
    .parquet(SILVER_PATH + "census_income/")
print("✅ census_income saved")

# Zillow Rent Silver
zillow_rent.write \
    .mode("overwrite") \
    .parquet(SILVER_PATH + "zillow_rent/")
print("✅ zillow_rent saved")

airbnb_listings_enriched.write \
    .mode("overwrite") \
    .parquet(SILVER_PATH + "airbnb_listings_enriched/")
print("✅ airbnb_listings_enriched saved")

print("\n🎉 Silver layer saved to MinIO!\n")

💾 SAVING SILVER LAYER TO MINIO
✅ census_population saved
✅ census_income saved
✅ zillow_rent saved
✅ airbnb_listings_enriched saved

🎉 Silver layer saved to MinIO!



In [72]:
# ============================================================
# 5. GOLD — MARKET ANALYSIS
#
# Il Gold layer combina tutti i dataset Silver per produrre
# le metriche finali che alimentano la dashboard.
# Tre tabelle Gold:
#   1. market_rental_stress: stress economico per ZIP code
#   2. airbnb_borough_summary: concentrazione Airbnb per borough
#   3. airbnb_vs_market: confronto Airbnb vs affitto di mercato
# ============================================================
print("=" * 55)
print("🏆 STEP 3 — GOLD")
print("=" * 55)


🏆 STEP 3 — GOLD


In [73]:
# ============================================================
# 5.1 GOLD — ECONOMIC PROFILE PER ZIP CODE
#
# Uniamo i tre dataset Silver (Census Population, Census Income, Zillow)
# usando zip_code come chiave di JOIN.
# Il risultato è un profilo economico completo per ogni ZIP code di NY.
# ============================================================
print("=" * 55)
print("🏆 STEP 3 — GOLD")
print("=" * 55)

# Inner join: teniamo solo ZIP code presenti in TUTTI e tre i dataset
# Se un ZIP manca in uno dei tre → viene scartato
economic_profile = (
    census_population
    .join(census_income, "zip_code", "inner")   # aggiunge median_income
    .join(zillow_rent, "zip_code", "inner")      # aggiunge market_rent
)

print(f"  ZIP con dati completi (NY totale): {economic_profile.count():,}")

  ZIP con dati completi (NY totale): 353
  ZIP solo 5 Borough NYC:            145


In [76]:
# ============================================================
# 5.3 GOLD — RENTAL STRESS INDEX
#
# Calcoliamo il "rent burden" per ogni ZIP code:
#   rent_burden_pct = (affitto_mensile * 12 / reddito_annuo) * 100
#
# Questo è lo standard HUD (Dept. of Housing and Urban Development):
#   < 30%  → Affordable   (spende meno del 30% del reddito in affitto)
#   30-50% → Stressed     (cost-burdened)
#   >= 50% → Severely Stressed (severely cost-burdened)
#
# È la metrica principale del progetto — identifica le "Stressed Areas"
# ============================================================

gold_market_analysis = (
    economic_nyc
    .withColumn(
        "rent_burden_pct",
        when(
            col("median_income") > 0,
            spark_round(
                # market_rent è mensile → x12 per annualizzarlo
                # dividiamo per reddito annuo e moltiplichiamo x100 per %
                (col("market_rent") * 12 / col("median_income")) * 100, 2
            )
        ).otherwise(None)  # NULL se reddito = 0 (evita divisione per zero)
    )
    .withColumn(
        # Assegna categoria di stress in base alla soglia HUD
        "stress_category",
        when(col("rent_burden_pct") >= 50, "🔴 Severely Stressed")
        .when(col("rent_burden_pct") >= 30, "🟡 Stressed")
        .when(col("rent_burden_pct").isNotNull(), "🟢 Affordable")
        .otherwise("⚪ No Data")  # ZIP senza dati sufficienti
    )
)

print("📊 Distribuzione Rental Stress:")
gold_market_analysis.groupBy("stress_category") \
    .count() \
    .orderBy("stress_category") \
    .show()

📊 Distribuzione Rental Stress:
+--------------------+-----+
|     stress_category|count|
+--------------------+-----+
|🔴 Severely Stressed|   38|
|         🟡 Stressed|   92|
|       🟢 Affordable|   15|
+--------------------+-----+



In [77]:
# ============================================================
# 5.4 GOLD — AIRBNB BOROUGH SUMMARY
#
# Aggreghiamo i listing Airbnb per borough per capire:
#   - Quanti listing ci sono (listing concentration)
#   - Quanto sono occupati in media (occupancy pressure)
#   - Quanti sono "Entire home" (appartamenti sottratti al mercato)
#   - Quanti host hanno più listing (proxy host commerciali)
#
# Non abbiamo avg_price_per_night (NULL nel dataset NY)
# quindi lavoriamo solo con occupancy e concentrazione.
# ============================================================

airbnb_borough_summary = (
    airbnb_listings_enriched
    # Teniamo solo listing con dati di occupancy validi
    .filter(col("final_occupancy_rate").isNotNull())
    .groupBy("neighbourhood_group_cleansed")
    .agg(
        # Numero totale listing per borough
        count("id").alias("num_listings"),

        # Occupancy media in percentuale (0-100%)
        spark_round(avg("final_occupancy_rate") * 100, 1).alias("avg_occupancy_pct"),

        # Media dei listing per host
        # Un valore alto indica presenza di host commerciali (multi-property)
        spark_round(avg("calculated_host_listings_count"), 1).alias("avg_host_listings"),

        # Numero di listing "Entire home/apt"
        # Questi sono appartamenti interi sottratti al mercato residenziale
        spark_sum(
            when(col("room_type") == "Entire home/apt", 1).otherwise(0)
        ).alias("entire_home_count")
    )
    .withColumn(
        # % di listing che sono appartamenti interi
        # Alto = più pressione sul mercato residenziale
        "entire_home_pct",
        spark_round(col("entire_home_count") / col("num_listings") * 100, 1)
    )
    .orderBy(col("num_listings").desc())
)

print("🏘️  Airbnb per Borough:")
airbnb_borough_summary.show(truncate=False)

🏘️  Airbnb per Borough:
+----------------------------+------------+-----------------+-----------------+-----------------+---------------+
|neighbourhood_group_cleansed|num_listings|avg_occupancy_pct|avg_host_listings|entire_home_count|entire_home_pct|
+----------------------------+------------+-----------------+-----------------+-----------------+---------------+
|Manhattan                   |4733        |39.5             |49.9             |3016             |63.7           |
|Brooklyn                    |3758        |45.0             |11.4             |1893             |50.4           |
|Queens                      |1757        |43.2             |22.1             |627              |35.7           |
|Bronx                       |361         |40.8             |4.3              |133              |36.8           |
|Staten Island               |133         |40.7             |2.7              |59               |44.4           |
+----------------------------+------------+-----------------+---

In [78]:
# ============================================================
# 5.5 GOLD — AIRBNB PRESSURE INDEX
#
# Combina listing concentration e occupancy rate per calcolare
# un indice di pressione speculativa per borough.
#
# Formula:
#   pressure_score = (num_listings / max_listings) * avg_occupancy_pct
#
# Interpretazione:
#   - num_listings / max_listings → concentrazione normalizzata (0-1)
#   - * avg_occupancy_pct         → pesa per quanto sono effettivamente usati
#
# Borough con tanti listing molto occupati = alta pressione sul mercato
# ============================================================

# Calcoliamo il massimo numero di listing tra tutti i borough
max_listings = airbnb_borough_summary \
    .agg(spark_max("num_listings")) \
    .collect()[0][0]

airbnb_pressure = (
    airbnb_borough_summary
    .withColumn(
        "pressure_score",
        spark_round(
            (col("num_listings") / max_listings) * col("avg_occupancy_pct"), 2
        )
    )
    .orderBy(col("pressure_score").desc())
)

print("📊 Airbnb Pressure Index per Borough:")
airbnb_pressure.show(truncate=False)

📊 Airbnb Pressure Index per Borough:
+----------------------------+------------+-----------------+-----------------+-----------------+---------------+--------------+
|neighbourhood_group_cleansed|num_listings|avg_occupancy_pct|avg_host_listings|entire_home_count|entire_home_pct|pressure_score|
+----------------------------+------------+-----------------+-----------------+-----------------+---------------+--------------+
|Manhattan                   |4733        |39.5             |49.9             |3016             |63.7           |39.5          |
|Brooklyn                    |3758        |45.0             |11.4             |1893             |50.4           |35.73         |
|Queens                      |1757        |43.2             |22.1             |627              |35.7           |16.04         |
|Bronx                       |361         |40.8             |4.3              |133              |36.8           |3.11          |
|Staten Island               |133         |40.7             

In [79]:
# ============================================================
# 5.6 GOLD — SALVATAGGIO SU MINIO
#
# Salviamo le tre tabelle Gold in formato Parquet su MinIO.
# coalesce(1) forza Spark a scrivere un singolo file Parquet
# invece di tanti file partizionati → più facile da leggere
# con pandas nella dashboard Streamlit.
# ============================================================
print("\n💾 Saving Gold layer to MinIO...")

# Stress economico per ZIP code → usato per la heatmap della dashboard
gold_market_analysis.coalesce(1).write \
    .mode("overwrite") \
    .parquet(GOLD_PATH + "market_rental_stress/")
print("✅ market_rental_stress saved")

# Concentrazione Airbnb per borough → grafico a barre dashboard
airbnb_borough_summary.coalesce(1).write \
    .mode("overwrite") \
    .parquet(GOLD_PATH + "airbnb_borough_summary/")
print("✅ airbnb_borough_summary saved")

# Pressure index → ranking borough per stress Airbnb
airbnb_pressure.coalesce(1).write \
    .mode("overwrite") \
    .parquet(GOLD_PATH + "airbnb_pressure/")
print("✅ airbnb_pressure saved")

print("\n🎉 Gold layer saved to MinIO!\n")


💾 Saving Gold layer to MinIO...
✅ market_rental_stress saved
✅ airbnb_borough_summary saved
✅ airbnb_pressure saved

🎉 Gold layer saved to MinIO!



In [81]:
# ============================================================
# 6. OUTPUT FINALE — RISULTATI
#
# Stampiamo un riepilogo dei risultati principali della pipeline.
# Questi sono i numeri che presenteremo nella dashboard e
# nella relazione dell'esame.
# ============================================================
print("=" * 55)
print("📊 RISULTATI FINALI")
print("=" * 55)

# Distribuzione ZIP code per categoria di stress
total_zip  = gold_market_analysis.count()
affordable = gold_market_analysis.filter(col("rent_burden_pct") < 30).count()
stressed   = gold_market_analysis.filter(
    (col("rent_burden_pct") >= 30) & (col("rent_burden_pct") < 50)).count()
severe     = gold_market_analysis.filter(col("rent_burden_pct") >= 50).count()

print(f"\n📍 ZIP code NYC analizzati: {total_zip}")
print(f"   🟢 Affordable  (<30%): {affordable:,}")
print(f"   🟡 Stressed  (30-50%): {stressed:,}")
print(f"   🔴 Severely    (≥50%): {severe:,}")

# Top 10 ZIP code più stressati — le "Stressed Areas" del progetto
print("\n🔝 TOP 10 ZIP CODE PER RENTAL STRESS:")
gold_market_analysis \
    .select(
        "zip_code",
        "total_population",
        "median_income",
        "market_rent",
        "rent_burden_pct",
        "stress_category"
    ) \
    .orderBy(col("rent_burden_pct").desc()) \
    .show(10, truncate=False)

# Top 5 ZIP più accessibili — per bilanciare l'analisi
print("\n🟢 TOP 5 ZIP PIÙ ACCESSIBILI:")
gold_market_analysis \
    .select("zip_code", "median_income", "market_rent", "rent_burden_pct") \
    .filter(col("rent_burden_pct").isNotNull()) \
    .orderBy(col("rent_burden_pct").asc()) \
    .show(5, truncate=False)

# Pressione Airbnb per borough
print("\n🏘️  AIRBNB PRESSURE INDEX PER BOROUGH:")
airbnb_pressure.show(truncate=False)

# Dettaglio per tipo di stanza
print("\n🏘️  AIRBNB: DETTAGLIO PER ROOM TYPE:")
airbnb_listings_enriched \
    .filter(col("final_occupancy_rate").isNotNull()) \
    .groupBy("room_type") \
    .agg(
        count("id").alias("num_listings"),
        spark_round(avg("final_occupancy_rate") * 100, 1).alias("avg_occupancy_pct")
    ) \
    .orderBy(col("num_listings").desc()) \
    .show(truncate=False)

📊 RISULTATI FINALI

📍 ZIP code NYC analizzati: 145
   🟢 Affordable  (<30%): 15
   🟡 Stressed  (30-50%): 92
   🔴 Severely    (≥50%): 38

🔝 TOP 10 ZIP CODE PER RENTAL STRESS:
+--------+----------------+-------------+-----------+---------------+--------------------+
|zip_code|total_population|median_income|market_rent|rent_burden_pct|stress_category     |
+--------+----------------+-------------+-----------+---------------+--------------------+
|10454   |39570           |24086        |2954.0105  |147.17         |🔴 Severely Stressed|
|10002   |76873           |48386        |4345.4795  |107.77         |🔴 Severely Stressed|
|10456   |87533           |34954        |2866.9722  |98.43          |🔴 Severely Stressed|
|10029   |77447           |38695        |3133.6191  |97.18          |🔴 Severely Stressed|
|10460   |59396           |36309        |2925.7778  |96.7           |🔴 Severely Stressed|
|10451   |50942           |38770        |2896.5452  |89.65          |🔴 Severely Stressed|
|10458   |7489